# Bitcoin Time Series Prediction with LSTM

#### Import necessary library needed for the model training

In [ ]:
from math import sqrt
from numpy import concatenate
from matplotlib import pyplot
import pandas as pd
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
import plotly.offline as py
import plotly.graph_objs as go
import numpy as np
import seaborn as sns
py.init_notebook_mode(connected=True)
%matplotlib inline

#### Read data set

In [ ]:
df = pd.read_csv(filepath_or_buffer="/kaggle/input/btcusdt-1h/btcusdt_1h.csv", index_col="datetime")

In [ ]:
df.head()

#### Check latest date of data

In [ ]:
df.tail()

#### Plot line graph base on `Weighted Price`

In [ ]:
def calculate_vwap(data):
    data['TP'] = (data['high'] + data['low'] + data['close']) / 3  # Typical Price
    data['TPV'] = data['TP'] * data['volume']  # Typical Price * Volume
    data['Cumulative TPV'] = data['TPV'].cumsum()
    data['Cumulative Volume'] = data['volume'].cumsum()
    data['VWAP'] = data['Cumulative TPV'] / data['Cumulative Volume']
    data.drop(['TP', 'TPV', 'Cumulative TPV', 'Cumulative Volume'], axis=1, inplace=True)  # Optional: Remove unnecessary columns

    return data


data = calculate_vwap(df)


In [ ]:
def signal(timeseries):
    
    signals=[0 for i in range(len(timeseries))]
    for i in range(0,len(timeseries)-1):
        if (timeseries[i+1]>timeseries[i]) :
            signals[i]=1
        elif (timeseries[i+1]<timeseries[i]) :
            signals[i]=-1
    return signals

In [ ]:
def returns(signals,timeseries):
    long=[]
    long_price=[]
    short_price=[]
    short=[]
    for i in range(len(signals)):
        if signals[i]==1:
            long.append(i)
            long_price.append(timeseries[i])
        elif signals[i]==-1:
            short.append(i)
            short_price.append(timeseries[i])
    cum_pnl=0
    while len(long)!=0 and len(short)!=0:
        if short[0]<long[0]:
            while short[0]<long[0]:
                cum_pnl+=(short_price[0]-long_price[0])*investment/short_price[0]-0.1/100*investment
                short.pop(0)
                short_price.pop(0)
            long.pop(0)
            long_price.pop(0)            
        elif long[0]<short[0]:
            while short[0]>long[0]:
                cum_pnl+=(short_price[0]-long_price[0])*investment/long_price[0]-0.1/100*investment
                long.pop(0)
                long_price.pop(0)
            short.pop(0)
            short_price.pop(0) 
    return cum_pnl       

In [ ]:
timeseries = df[["close"]].values.astype('float32')

In [ ]:
btc_trace = go.Scatter(x=data.index, y=data['close'], name= 'Price')
py.iplot([btc_trace])

#### Use MinMaxScaler to normalize `VWAP` to range from 0 to 1

In [ ]:
from sklearn.preprocessing import MinMaxScaler
values = data['close'].values.reshape(-1,1)
print(values)
values = values.astype('float32')
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)
print(scaled)

#### Split 70% of data for training and 30% for testing

In [ ]:
train_size = int(len(scaled) * 0.7)
test_size = len(scaled) - train_size
train, test = scaled[0:train_size,:], scaled[train_size:len(scaled),:]
print(len(train), len(test))

#### Create function for creating dataset with look back

In [ ]:
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset) - look_back):
        a = dataset[i:(i + look_back), 0]
        dataX.append(np.array(a))
        dataY.append(dataset[i + look_back, 0])
    return np.array(dataX), np.array(dataY)

#### Generate dataset for trainX, trainY, testX, testY

In [ ]:
look_back = 10
train_X,trainY = create_dataset(train, look_back)
train_Y= signal(train[look_back:])
test_X, testY = create_dataset(test, look_back)
test_Y= signal(test[look_back:])

#### Reshape X for model training

In [ ]:
trainX = np.reshape(train_X, (train_X.shape[0], 1, train_X.shape[1]))
testX = np.reshape(test_X, (test_X.shape[0], 1, test_X.shape[1]))

#### Running the LSTM model with 300 epochs

In [ ]:
def custom_loss(ypred,yreal):
    sig=signal(ypred)
    sig2=signal(yreal)
    l1=keras.losses.MSE(y_true, y_pred)
    loss = keras.losses.CategoricalCrossentropy()
    l2=loss(sig,sig2)
    return l1+l2
    
    
model1 = Sequential()
model1.add(LSTM(100, input_shape=(trainX.shape[1], trainX.shape[2])))
model1.add(Dense(1))
model1.compile(loss="mse", optimizer='adam')
history = model1.fit(trainX, trainY, epochs=300, batch_size=len(trainX), validation_data=(testX, testY), verbose=0, shuffle=False)


#### Plot line graph to show amount loss according the the epoch

In [ ]:
pyplot.plot(history.history['loss'], label='train')
pyplot.plot(history.history['val_loss'], label='test')
pyplot.legend()
pyplot.show()

#### Make prediction using textX and plotting line graph against testY

In [ ]:
yhat = model1.predict(testX)
pyplot.plot(yhat, label='predict')
pyplot.plot(testY, label='true')
pyplot.legend()
pyplot.show()

#### Scaler Inverse Y back to normal value

#### Plot line graph with Y as USD

In [ ]:
yhat_inverse = scaler.inverse_transform(yhat.reshape(-1, 1))
testY_inverse = scaler.inverse_transform(testY.reshape(-1, 1))
rmse = sqrt(mean_squared_error(testY_inverse, yhat_inverse))
print('Test RMSE: %.3f' % rmse)
pyplot.plot(yhat_inverse, label='predict')
pyplot.plot(testY_inverse, label='actual', alpha=0.5)
pyplot.legend()
pyplot.show()